<a href="https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZeroGrad on Titanic: check every number yourself

Predict who survived the Titanic **without training a model**. ZeroGrad compares pretrained tabular foundation models on your labeled history, keeps the best one (the *champion*) and scores new rows with it.

Everything here runs on your machine (or this Colab), with no account and no API key. We keep our own holdout aside, so at the end we check ZeroGrad's numbers ourselves.

In [ ]:
!pip install -q zerograd

## 1. Data

The OpenML version of Titanic has 1,309 passengers. We keep 20% aside as our own holdout: no model ever sees it.

`boat` and `body` are only known *after* the sinking (who got on a lifeboat, whose body was found), so they would leak the answer. `home.dest` is free text. We drop the three.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

df = fetch_openml("titanic", version=1, as_frame=True).frame
df = df.drop(columns=["boat", "body", "home.dest"])
train, holdout = train_test_split(df, test_size=0.2, stratify=df["survived"], random_state=0)
len(train), len(holdout)

## 2. The championship

One call. ZeroGrad profiles the columns (drops identifiers and free text such as `name` and `ticket`), holds out 20% of `train`, scores **every** holdout row with each model and keeps the best. The first run downloads each model's weights from its official repository (once).

In [ ]:
from zerograd import ZeroGrad

zg = ZeroGrad()   # local: no API key
cap = zg.create("titanic-survival", examples=train, target="survived")
cap.evaluation

In [ ]:
cap.baselines   # the same holdout without a model: random scores and always the majority class

In [ ]:
cap.columns     # which columns were used, and why the others were dropped

## 3. Predict our own holdout and check the numbers

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

preds = cap.predict(holdout.drop(columns="survived"))
y = holdout["survived"].astype(int)
print("ROC AUC ", round(roc_auc_score(y, preds["score"]), 3))
print("Accuracy", round(accuracy_score(y, preds["prediction"].astype(int)), 3))
preds.head()

In [ ]:
# One passenger, as an app would call it: a dict in, a dict out
passenger = holdout.drop(columns="survived").iloc[0].to_dict()
cap.predict(passenger)

## 4. How much history do you need?

Foundation models read their context (the labeled rows) on every prediction, so a smaller context answers faster. `context_curve` runs the champion on growing samples of its own context and scores each on our holdout. The recommended size is the smallest one within 0.005 of the best ROC AUC.

In [ ]:
curve = cap.context_curve(holdout, sizes=[50, 100, 200, 400, 800])
curve

In [ ]:
curve.plot();

In [ ]:
cap.update(context_rows=curve.recommended)   # predict with the smaller, faster context from now on
len(cap.context_y)

## 5. Multiclass: Iris

The same calls work for three classes.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True).frame
iris_cap = zg.create("iris", examples=iris.sample(120, random_state=0), target="target")
iris_cap.predict(iris.drop(columns="target").tail(3))

## 6. Keep it, or run it in production

`cap.save("titanic.zg")` writes the capability to one file (no weights inside) and `zerograd.load` opens it anywhere the SDK is installed. The file holds a sample of your rows as the model's context: treat it like the data.

To run it in production with versions, scheduled championships, monitoring and Explain, the same code runs on the ZeroGrad platform with `ZeroGrad(remote=True)`: see [zerograd.dev](https://zerograd.dev).

In [ ]:
cap.save("titanic.zg")